<a href="https://colab.research.google.com/github/mailtoasif/Data-Science/blob/main/tts_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# Fast install, might break in the future.
# !pip install 'sphn<0.2'
# !pip install --no-deps "moshi==0.2.11"
# Slow install (will download torch and cuda), but future proof.
!pip install "moshi==0.2.11"

In [18]:
import numpy as np
import torch # it is used for the neural networks
from moshi.models.loaders import CheckpointInfo # checkpoints class from moshi model . its loads weights and states of trained model
from moshi.models.tts import DEFAULT_DSM_TTS_REPO, DEFAULT_DSM_TTS_VOICE_REPO, TTSModel

from IPython.display import display, Audio

In [12]:
# Configuration
text = (
    "Why don’t you ever listen? I am so tired of repeating myself again and again! "  # Angry
    "I miss those days so much... nothing feels the same anymore. "  # Sad
    "Wait… did you hear that sound? Something is moving in the dark!"  # Fear
)
voice = "expresso/ex03-ex01_happy_001_channel1_334s.wav"
print(f"See https://huggingface.co/{DEFAULT_DSM_TTS_VOICE_REPO} for available voices.")


See https://huggingface.co/kyutai/tts-voices for available voices.


In [13]:
# Set everything up
checkpoint_info = CheckpointInfo.from_hf_repo(DEFAULT_DSM_TTS_REPO)

# Check for CUDA availability and set the device accordingly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tts_model = TTSModel.from_checkpoint_info(
    checkpoint_info, n_q=32, temp=0.6, device=device
)

# If you want to make a dialog, you can pass more than one turn [text_speaker_1, text_speaker_2, text_2_speaker_1, ...]
entries = tts_model.prepare_script([text], padding_between=1)
voice_path = tts_model.get_voice_path(voice)
# CFG coef goes here because the model was trained with CFG distillation,
# so it's not _actually_ doing CFG at inference time.
# Also, if you are generating a dialog, you should have two voices in the list.
condition_attributes = tts_model.make_condition_attributes([voice_path], cfg_coef=2.0)

Using device: cpu


In [ ]:
print("Generating audio...")

pcms = []


def _on_frame(frame):
    print("Step", len(pcms), end="\r")
    if (frame != -1).all():
        pcm = tts_model.mimi.decode(frame[:, 1:, :]).cpu().numpy()
        pcms.append(np.clip(pcm[0, 0], -1, 1))


# You could also generate multiple audios at once by extending the following lists.
all_entries = [entries]
all_condition_attributes = [condition_attributes]
with tts_model.mimi.streaming(len(all_entries)):
    result = tts_model.generate(
        all_entries, all_condition_attributes, on_frame=_on_frame
    )

print("Done generating.")
audio = np.concatenate(pcms, axis=-1)

Generating audio...


In [ ]:
display(Audio(audio, rate=tts_model.mimi.sample_rate, autoplay=True))

In [ ]:
from IPython.display import Audio, display
import soundfile as sf

# Save audio to a file
sf.write("tts_output.wav", audio, tts_model.mimi.sample_rate)

# Play inside notebook
display(Audio("tts_output.wav", rate=tts_model.mimi.sample_rate, autoplay=True))

print("Audio saved as tts_output.wav — you can download it from your notebook files.")
